# **03 - Machine Learning Model EDA**
---

## <a id = 'summary'></a>**Summary**

---

## **Table of Content**

[Summary](#summary)

[Objective](#obj)

[Dataset Info](#data-info)

[Model Exploration Strategy](#strategy)

[Initial Setup](#setup)

[XGBoost](#xgboost)

---

## <a id ='obj'></a>**Objectives**
To evaluate and compare XGBoost, SARIMA, and CNN models under different configurations and feature selection strategies, while incorporating conformal prediction to quantify prediction uncertainty and assess the reliability of forecasting intervals.


## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## <a id = 'strategy'></a>**Model Exploration Strategy**
We're explore 3 models:
1. eXtreme Gradient Boosting (XGBoost)
2. Seasonal Autoregressive Integrated Moving Average (SARIMA)
3. Convolutional Neural Network (CNN)

And as these models are all of different model family (*gradient boosting, statistical and deep learning family, respectively)*, they have to be treated very differently. As such, we'll explore these models sequentially.

For every model exploration, we'll keep it within these stages:
1. Feature preprocessing
2. Hyperparameter Tuning

, which will be tailored to fit the currently explored model. Description for each model stages can be found on top of sections named after the model.

## **Initial Setup**

### Import Necessary Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from sklearn.preprocessing import LabelEncoder
import numpy as np
from xgboost import XGBRegressor
import optuna
from sklearn.model_selection import cross_val_score
import joblib
from sklearn.model_selection import train_test_split
from pmdarima import auto_arima

# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy, feature_engineering


### Import Raw Dataset

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

### Data Preprocessing and Formation of Feature Space

In [ ]:
# Data preprocessing
energy = preprocess_energy(energy)

# Form feature through feature engineering
energy = feature_engineering(energy)

In [ ]:
# Extract features and target
X = energy.drop(columns = ['Appliances'])
y = energy['Appliances']

## <a id = 'xgboost'></a>**XGBoost**

### Stages Decsription
**Feature preprocessing**

These actions will be done to accomodate XGBoost model:
* Cyclical time encoding
    
    **Justification**: Represents the circular nature of time, so 23:00 and 00:00 are treated as nearby times.

* Encode categorical features

    **Justification**: month is in str data type, which is incompatible with optuna.

* Time-based train-test split

    **Justification**: As this is a time-series dataset, where row order means something, we will split by chronological order rather than shuffling.

**Hyperparameter Tuning**

These hyperparameters will be searched using **Bayesian Optimization**:
* 'max_depth': [3, 5, 7, 9]
* 'min_child_weight': [1, 3, 5, 7]
* 'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
* 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
* 'learning_rate': [0.01, 0.05, 0.1, 0.2]

### Feature Preprocessing

*Cyclical Time Encoding*

In [ ]:
# Create copy to isolate preprocessing
X_xgb = X.copy()

In [ ]:
# Specify target column to apply cyclical encoding
col = 'hour'

# Apply both sine and cosine cyclical encoding
X_xgb[f"{col}_sin"] = np.sin(2 * np.pi * X_xgb[col] / 24 * 2)
X_xgb[f"{col}_cos"] = np.cos(2 * np.pi * X_xgb[col] / 24 * 2)

# Drop the original raw column
X_xgb = X_xgb.drop(columns=[col])

*Encode categorical features*

In [ ]:
# Encode month
le_month = LabelEncoder()
X_xgb['month'] = le_month.fit_transform(X_xgb['month'])

# Convert returned numpy array into DataFrame (for compatibility)
X_xgb = pd.DataFrame(X_xgb)

*Time-based Train-Test Split (80/20)*

In [ ]:
# 80-20 train-test split, no shuffling (to preserve  chronlogical order)
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb, y, test_size=0.2, random_state=42, shuffle = False
)

### Hyperparameter Tuning

In [ ]:
def objective(trial):

    # Disable Optuna's verbose logging to keep notebook output clean
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # Define hyperparameter grid
    params = {
        'max_depth' : trial.suggest_int('max_depth', 2, 8),
        'min_child_weight': trial.suggest_int('min_child_weight',1, 10) ,
        'subsample':trial.suggest_float('subsample', 0.6, 0.9) ,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9) ,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2) ,
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        
        'random_state': 42
    }

    
    # Initialize the model with suggested parameters
    xgb = XGBRegressor(**params)
    
    # Evaluate using Cross-Validation
    score = cross_val_score(
        xgb,
        X_train_xgb,
        y_train_xgb, 
        cv=3,
        scoring = 'neg_root_mean_squared_error'
    ).mean()
    
    # Return the metric to maximize
    return score



In [ ]:
# Initiate search
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

In [ ]:
# Print best results
print(f"Best RMSE: {-study.best_value:.2f}")

# Print best model parameter
best_params = study.best_params
print('\n==BEST PARAMETERS=')
for item, value in best_params.items():
    print(f'{item} = {value}')

In [ ]:
# Train model using best parameter
xgb_best = XGBRegressor(
    **best_params,
    random_state=42
)

xgb_best.fit(X_train_xgb, y_train_xgb)

In [ ]:
joblib.dump(xgb_best, f"{PROJECT_ROOT}/models/xgb_best.pkl")

## <a id = 'sarima'></a>**SARIMA**

### Stages Decsription
**Feature preprocessing**

These actions will be done to accomodate SARIMA:

* Change 'month' to datetime

* Augmented Dickey-Fuller (ADF) test

    **Justification**: SARIMA assumes series stationarity, so this test checks it formally. If our series is not stationary (or in another words, the mean and variance changes over time), we'll have to perform differencing.

**Hyperparameter Tuning**

The hyperparameter tuning for this model is more straightforward, as we only need to call 'auto_arima' method from 'pmdarima' package to automate the process.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.deterministic import Fourier

In [ ]:
# Isolate first
X_sarima = X.copy()

*Encode categorical feature*

In [ ]:
# Encode month
X_sarima['month'] = le_month.fit_transform(X_sarima['month'])

# Convert returned numpy array into DataFrame (for compatibility)
X_sarima = pd.DataFrame(X_sarima)

#### Feature Preprocessing
*ADF Test*

• **Null Hypothesis** ($H₀$): The time series is non-stationary.

• **Alternative Hypothesis** ($H₁$): The time series is stationary.

In [ ]:
# ADF Test
result = adfuller(y, autolag='AIC', result_object = True)
print(f'p-value: {result[1]}')   


Since the p-value ≤  0.05, we reject the $H₀$. This series is indeed stationary, so no we can set D (differencing order) = 0

*Fourier Transform*

In [ ]:
# Apply Fourier transform
f = Fourier(period=48, order=3)
fourier_df = f.in_sample(y.index)

# Combine the Fourier terms with original data
X_sarima = pd.concat([X_sarima, fourier_df], axis=1)

*Train-test Split*

In [ ]:
X_train_sarima, X_test_sarima, y_train_sarima, y_test_sarima = train_test_split(
    X_sarima, y, test_size=0.2, shuffle=False
)

#### Hyperparameter Tuning

*auto_arima hyperparameter search*

In [ ]:
# Start best hyperparameter search
best_auto_model = auto_arima(
    y=y_train_sarima,
    X=X_train_sarima,
    d=0, D = 0,          
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    m=0, 
    seasonal=False,
    stepwise=True,  
    trace=True,     
    error_action="ignore",
    suppress_warnings=True
)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
y_pred_sarima = best_auto_model.predict(
    n_periods=len(y_test_sarima), 
    X=X_test_sarima
)

rmse = np.sqrt(mean_squared_error(y_test_sarima, y_pred_sarima))

print(f"Test RMSE: {rmse:.4f}")

In [ ]:
joblib.dump(best_auto_model, f"{PROJECT_ROOT}/models/sarima_best.pkl")

## <a id = 'cnn'></a>**CNN**

### Stages Decsription
**Feature preprocessing**

**Hyperparameter Tuning**
